In [ ]:
"""
Example Code for training an MRI Image Reconstruction Network from Synthetic Fractals

Provided code includes trajectories, model training and pre-trained models as implemented for the paper.

@author: Anirudh Raman
"""

import numpy as np
import tensorflow as tf
import tensorflow_mri as tfmri
from pathlib import Path
from sklearn.model_selection import train_test_split
import glob
import matplotlib.pyplot as plt
from matplotlib import animation
from keras.callbacks import EarlyStopping, ModelCheckpoint


import utils.fractal_utils as fr 
import utils.preprocessing_natural_videos as preproc_video
import utils.preprocessing_trajectory_gen as preproc_traj
import utils.preprocessing_multicoil_noselect as preproc_multicoil


### 1. Build fractal dataset

#### 1.1. Configure processing pipeline

In [ ]:
config_preproc=preproc_multicoil.config_base_preproc()
config_preproc['normalize_input'] = True

config_natural_images=preproc_video.config_default(config_preproc['base_resolution'],config_preproc['phases'])
config_natural_images['num_coils'] = [15]
config_natural_images['complex_transform'] = 1
              
config_traj=preproc_traj.config_radial_traj()
config_traj['base_resolution'] = 256 
config_traj['radial_spokes'] = 13
config_traj['ordering'] = 'sorted_half'
    

print('Config natural image to kspace:',config_natural_images,
            '\nConfig preprocessing:',config_preproc,
            '\nConfig traj:',config_traj)
    

preproc_natural_image=preproc_video.preprocessing_fn(**config_natural_images)
preproc_function=preproc_multicoil.preprocessing_fn(**config_preproc)
roll_function=preproc_multicoil.rolling_fn(phases=24,rotation=1,input_format=config_preproc['input_format'], output_format=config_preproc['output_format'])
traj_function=preproc_traj.create_traj_fn(**config_traj)

#### 1.2. Generate fractals and process using natural videos pipeline

In [ ]:
processed_data_path = '/workspaces/fractal_github/data' # directory to store training data
n_data = 500 # number of datasets to generate

Path(processed_data_path).mkdir(parents=True, exist_ok=True)

# Get candidate values for c by generating a Mandelbrot set 
C_grid = fr.build_grid([1.5]*4, [100]*4)
mandelbrot_vals = fr.mandelbrot_escape(C_grid, max_iter=50)
ids = np.argwhere((mandelbrot_vals<30) & (mandelbrot_vals>10))
C_candidates = C_grid[tuple(ids.T)]

# Build a grid for Julia sets [X,Y,_,T]
grid = fr.build_grid(limits=[1,1,0,0.2], sizes=[256,256,1,32])
i=0
while i < n_data:
    print(i)
    idx = np.random.randint(0,C_candidates.shape[0])
    c = C_candidates[idx]     # Randomly select a c value 
    name = '_'.join([f'{i:.2f}' for i in c])
    escape_vals = np.squeeze(fr.julia_escape(grid, c, max_iter=50)) # Generate a Julia set 
    if fr.test_complexity(escape_vals): # Only progress if the fractal complexity is above a threshold
        complex_fractal = fr.make_complex_fractal(np.transpose(escape_vals,[2, 0, 1])) # Simulate 2 channels from the single-channel fractal
        video= {'video':tf.cast(tf.clip_by_value(complex_fractal,0,255),tf.uint8)}

        # process the fractals to get paired training data 
        kspace=preproc_natural_image(video)
        ds0=tf.data.Dataset.from_tensors(kspace)
        image=traj_function(ds0)
        for element in image:
            zfill, image=preproc_function(element)
            zfill, image=roll_function(zfill, image)
            np.save(f'{processed_data_path}/{name}_input.npy', zfill)
            np.save(f'{processed_data_path}/{name}_output.npy', image)
        
        i+=1

### 2. Data pipeline

#### 2.1. Define data generators

In [ ]:
files = np.unique(['_'.join(pat.split('/')[-1].split('_')[:-1]) for pat in glob.glob(f'{processed_data_path}/*.npy')])

train_files, temp_files = train_test_split(files, test_size = 0.25, random_state = 42)
val_files, test_files = train_test_split(temp_files, test_size = 0.6, random_state = 42)
print(f"{len(train_files)} training, {len(val_files)} validation, and {len(test_files)} test files.")


class CustomDataGen:
    def __init__(self, files, cohort):
        self.files = files

    def data_generator(self):
        for file in self.files:
            true_im_file_path = f'{processed_data_path}/{file}_output.npy'
            noisy_im_file_path = f'{processed_data_path}/{file}_input.npy'  # Specify your file path
            true_im= np.load(true_im_file_path)#[...,np.newaxis]
            noisy_im = np.load(noisy_im_file_path)
            yield noisy_im, true_im

    def get_gen(self):
        return self.data_generator


N_outputs = 1  # Number of output channels
#shape = [240,192,40,1]  # Input shape
input_shape = [None,None,None,20]   
output_shape = [None,None,None,1]  


# Create data generators for training and validation datasets
train_gen = CustomDataGen(train_files, 'train').get_gen()
val_gen = CustomDataGen(val_files, 'val').get_gen()

# Create TensorFlow datasets from the generators
output_signature = (tf.TensorSpec(shape=input_shape, dtype=tf.float32), 
                    tf.TensorSpec(shape=output_shape, dtype=tf.float32))

train_ds = tf.data.Dataset.from_generator(train_gen, output_signature=output_signature)
val_ds = tf.data.Dataset.from_generator(val_gen, output_signature=output_signature)

# Define the batch size
BATCH_SIZE = 1

# Prepare training dataset: shuffle, batch, and prefetch for optimized loading
train_ds = train_ds.shuffle(buffer_size=20, seed=42, reshuffle_each_iteration=True) \
                .batch(BATCH_SIZE) \
                .prefetch(tf.data.AUTOTUNE)

# Prepare validation dataset: batch and prefetch (no need to shuffle validation data)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


#### 2.2. Visualise the data 

In [ ]:
iterator = iter(train_ds)

noisy_im, true_im = next(iterator)

print(noisy_im.shape, true_im.shape)
plt.imshow(np.abs(np.squeeze(noisy_im)[0,...,0]),cmap='gray')
plt.show()
plt.imshow(np.abs(np.squeeze(true_im)[0]), cmap='gray')
plt.show()

### 3. Define the model and training parameters

In [ ]:
pretrained_weights=None # set to "<fname>" of weights file

inputs = tf.keras.Input(shape = input_shape)
config_model= {'filters': [64,92,128],
            'block_depth': 2,
            'kernel_size': 3,
            'activation': lambda x: tf.keras.activations.relu(x,alpha=0.1),
            'out_channels': 1,
            'kernel_initializer': tf.keras.initializers.HeUniform(seed=1)}
outputs=tfmri.models.UNet3D(**config_model)(inputs)

model = tf.keras.Model(inputs = inputs, outputs = outputs)

if pretrained_weights:
    model.load_weights(pretrained_weights)

model.compile(
    loss=tfmri.losses.SSIMLoss(image_dims=2),
    optimizer=tf.keras.optimizers.Adam(learning_rate = 1e-4),  # Adam optimizer for training
    metrics= [tfmri.metrics.PSNR(image_dims=2),tfmri.metrics.SSIM(image_dims=2)]  # Track accuracy during training and validation
)


### 4. Define a custom callback for training

In [ ]:
class CustomCallback(tf.keras.callbacks.Callback):
    def __init__(self, data_path, counter=0, save_every=50):
        super(CustomCallback, self).__init__()
        self.save_every = save_every  # Interval for epoch evaluations
        #self.save_every_batch = save_every_batch  # Interval for batch evaluations
        self.counter = counter          # Initialize epoch counter
        #self.batch_counter = batch_counter          # Initialize batch counter
        self.data_path = data_path      # Path to the dataset

    def on_epoch_end(self, epoch, logs=None):
        """Called at the end of each epoch."""
        self.counter += 1  # Increment the epoch counter
        if self.counter % self.save_every == 0:
            print(f"Evaluating model after epoch {self.counter}...")
            self.evaluate_model()  # Call the evaluation method


    def on_train_end(self, logs=None):
        """Called at the end of training."""
        print("Final evaluation of the model...")
        self.evaluate_model(subset=-1)  # Final evaluation after training completes
    
    def evaluate_model(self, cohorts=['test'],subset=10):
        # Iterate over each cohort
        for cohort in cohorts:
            ssims = []  # List to hold dice values for each file

            # Load the trained model
            #model = tf.keras.models.load_model(f'models/{model_name}.h5', compile=False)
            all_files = {'train': train_files,'val': val_files,'test': test_files}    
            # Iterate over each file in the cohort
            for file in all_files[cohort][:subset]:
                # Prepare the test data generator
                test_gen = CustomDataGen([file], cohort).get_gen()
                test_ds = tf.data.Dataset.from_generator(test_gen, output_signature=output_signature)
                test_ds = test_ds.batch(1).prefetch(-1)

                # Generate predictions for the file
                denoised_image = np.squeeze(model.predict(test_ds))

                # Create a directory for saving results if it doesn't exist
                Path(f'results/denoised_images').mkdir(parents=True, exist_ok=True)

                # Save the predicted image
                np.save(f'results/denoised_images/{file}.npy', denoised_image)

                noisy_image, true_image = next(iter(test_ds))
                true_image = np.squeeze(true_image)
                noisy_image = np.squeeze(convert_multicoil_to_image(noisy_image))

                def expand_for_ssim(img):
                    img= tf.expand_dims(tf.squeeze(img), axis=0)
                    img= tf.expand_dims(img, axis=-1)
                    return img

                ssim_val = 1.0 - float(tfmri.losses.ssim_loss(expand_for_ssim(true_image), expand_for_ssim(denoised_image), image_dims=3))
                
                ssims.append(ssim_val)  # Append ssim value to the list

                fig, axs = plt.subplots(1, 3, figsize=(6, 3))
                axs[0].set_title(f"True Image")
                axs[1].set_title(f"Denoised Image\nSSIM: {ssim_val:.2f}")
                axs[2].set_title("Undersampled Image")
                frames = []

                # Create frames for the animation
                for i in range(true_image.shape[0]):
                    # Clear previous titles
                    # Set main title for the figure with frame index
                    title_text = fig.text(0.5, 0.95, f"Frame {i+1}/{true_image.shape[0]}", ha='center')
                    
                    # Plot the images for this frame
                    p1 = axs[0].imshow(true_image[i,...], cmap='gray', vmin=np.min(true_image), vmax=np.max(true_image))
                    p2 = axs[1].imshow(denoised_image[i,...], cmap='gray', vmin=np.min(denoised_image), vmax=np.max(denoised_image))
                    p3 = axs[2].imshow(np.squeeze(noisy_image)[i,...], cmap='gray', vmin=np.min(noisy_image), vmax=np.max(noisy_image))
                    # Append the current frames to the list
                    frames.append([p1, p2, p3, title_text])

                # Create and save animation
                ani = animation.ArtistAnimation(fig, frames)
                Path(f'gifs/results/{cohort}').mkdir(parents=True, exist_ok=True)
                ani.save(f'gifs/results/{cohort}/{file}.gif', fps=20)
                plt.close()  # Close the plot to free memory

def convert_multicoil_to_image(multicoil):
    real_part, imag_part = tf.split(multicoil, 2, axis=-1)
    complex_tensor = tf.complex(real_part, imag_part)
    complex_image  = tfmri.coils.combine_coils(complex_tensor, coil_axis=-1)
    return tf.math.abs(complex_image)

### 5. Train the model

In [ ]:

# Define ModelCheckpoint callback
mc = ModelCheckpoint(
    f'models/fractal_unet.h5',  # Path to save the model
    save_best_only=False,        # Save only the best model
    monitor='loss',             # Monitor the training loss
    mode='min'                  # Save when the loss is minimized
)


# Define custom callback for additional evaluations or saving
eval_every_batch = CustomCallback(counter=0, save_every=25, data_path=processed_data_path)

# Fit the model with the specified training and validation datasets

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=200,                # Set the maximum number of epochs for training
    callbacks=[mc, eval_every_batch]  # Include all defined callbacks
)